<a href="https://colab.research.google.com/github/colbyrobbins/Gemma-Lie-Detector/blob/text-model/notebooks/03_text_deepdive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma Lie Detector - 03 Text deep dive
Notebook 01 treats *delete one word* occlusion as the reference for what drives Gemma's decision. That reference has blind spots: deleting a word can break the sentence, redundant words hide each other, and some words matter only together.

This notebook checks how far the reference can be trusted, using the same reviews and the citations already saved by notebook 01:

1. **Replace instead of delete** - swap each word for a placeholder.
2. **Integrated gradients** - a gradient-based method that looks inside the model, run with a zero-embedding baseline.
3. **Smallest flip set** - the fewest words whose removal flips the decision, and whether Gemma cited them.
4. **One behavioural judge for every method** - delete each method's top words and compare how fast the decision falls.

**Before running:** a GPU runtime, a Colab secret `HF_TOKEN`, and the core results file from notebook 01 in `results/text_results/` (committed in the repo, or on your Drive if you use the Drive cell).

In [ ]:
import os, sys
BRANCH = "text-model"   # branch to clone; change to "main" after merging
if not os.path.exists("src/gemma_lie_detector"):
    if not os.path.exists("Gemma-Lie-Detector"):
        !git clone -b {BRANCH} https://github.com/colbyrobbins/Gemma-Lie-Detector.git
    %cd Gemma-Lie-Detector
!pip -q install -U "transformers>=4.50" accelerate datasets scikit-learn tqdm scipy matplotlib
sys.path.insert(0, os.path.abspath("src"))   # makes the package importable without a kernel restart
assert os.path.exists("src/gemma_lie_detector"), "repo code not found: did you push it to GitHub (and is the repo public)?"
print("repo ready in", os.getcwd())


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))   # needs the Gemma license accepted on this HF account

In [ ]:
N = 300                          # reviews to analyse; raise it later to add a second batch (finished ones are skipped)
RESULTS_DIR = "results/text_results"
CORE_FILE = f"{RESULTS_DIR}/text_core_v2.jsonl"       # citations from notebook 01; rename if your file is named differently
DEEP_FILE = f"{RESULTS_DIR}/text_deepdive_v1.jsonl"   # this notebook's output (resumable)
MODEL_ID = "google/gemma-3-4b-it"
DTYPE = "auto"                   # bfloat16; see notebook 01

**Optional but recommended:** save results to Google Drive so a disconnect does not lose them. If you use it, put your core results file in that Drive folder first.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/gemma_lie_detector"
CORE_FILE = f"{RESULTS_DIR}/text_core_v2.jsonl"
DEEP_FILE = f"{RESULTS_DIR}/text_deepdive_v1.jsonl"

In [ ]:
from gemma_lie_detector.model import GemmaScorer
from gemma_lie_detector.prompts import SENTIMENT

if "scorer" not in globals():     # never load a second copy of Gemma when re-running for a bigger N
    scorer = GemmaScorer(SENTIMENT.labels, model_id=MODEL_ID, dtype=DTYPE)
print("dtype:", scorer.dtype)

**Reviews to analyse.** The same seed as notebook 01, so the first reviews are identical.

In [ ]:
from gemma_lie_detector.data import load_sst2
examples = load_sst2(n=N, seed=0)
print("examples:", len(examples))

## Intial test for integrated gradients
Run this before the long job. For two reviews it prints the words IG finds most supportive and the **completeness error**: IG's scores should add up to the change in margin between the baseline and the review. A relative error well below 0.1 means the numbers can be trusted; a large error means too few steps or too much bfloat16 noise (a pad baseline, `baselines=("pad",)`, failed this check in testing, so zero is the default).

In [ ]:
import numpy as np
from gemma_lie_detector.attribution import GemmaAttributor
attributor = GemmaAttributor(scorer, steps=64)   # zero baseline by default

for ex in examples[:2]:
    logp, _ = scorer.score([SENTIMENT.prompt(ex.render())])
    pred = int(np.argmax(logp[0]))
    print("REVIEW:", ex.render(), "| decision:", SENTIMENT.labels[pred])
    for baseline in attributor.baselines:
        scores, err = attributor.attribute(SENTIMENT, ex, pred, baseline=baseline)
        top = np.argsort(-scores)[:5]
        print(f"  {baseline:5s} completeness error {err:.3f} | top words:", [(ex.elements[i], round(float(scores[i]), 1)) for i in top])

## Run the comparison
Per review this runs several hundred short forward passes plus integrated gradients, so expect it to be slower than notebook 01's occlusion. It resumes if interrupted.

In [ ]:
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
from gemma_lie_detector.deepdive import run_deep
run_deep(examples, SENTIMENT, scorer, attributor, CORE_FILE, DEEP_FILE)

## Results

In [ ]:
from gemma_lie_detector.analysis_deep import (load_deep, summarize_deep, method_agreement,
                                              disagreement_cases, make_deep_figures)
deep = load_deep(DEEP_FILE)
print("reviews analysed:", len(deep))
display(summarize_deep(deep).round(3))
display(method_agreement(deep).round(2))
display(disagreement_cases(deep))
make_deep_figures(deep, outdir=f"{RESULTS_DIR}/figures_deepdive")

## Combining occlusion and integrated gradients
Neither method is ground truth, so claims are stronger when they hold under both. Everything below uses only reviews whose IG scores pass the completeness check (relative error under 0.1), and says how many that leaves.

- **Paired behaviour comparison:** Gemma's cited words against each method's top-m words on the same deletion test. `drop_diff` is the method's log-odds drop minus the cited words' drop (positive = the method's words hurt the decision more). `only_cited` / `only_method` count reviews where just one of them flips the decision; `flip_p` is the exact two-sided p-value for those counts.
- **Flip patterns:** which of cited / occlusion / IG flip the decision together. Rows where they disagree show where one method misses something.
- **Where each cited word ranks:** top-3 under occlusion, under IG, under both, or neither. Words in `neither` are the strongest candidates for decorative citations; words ranked high by only one method are inconclusive.

In [ ]:
from gemma_lie_detector.analysis_deep import (reliable_ig, behaviour_comparison, flip_patterns,
                                              cited_word_ranks, triangulation_table, method_agreement)
ok = reliable_ig(deep)          # IG completeness error < 0.1 (zero baseline)
print(f"reviews analysed: {len(deep)}; IG passes the completeness check on {len(ok)}")

display(behaviour_comparison(deep).round(3))
display(flip_patterns(deep).round(3))

words = cited_word_ranks(deep)
display(triangulation_table(words).round(3))
display(method_agreement(ok).round(2))     # rank agreement between methods, trusted-IG reviews only
display(words[words.group == "neither"].head(15))   # cited words neither method finds important

In [ ]:
# Save your work: download, or copy to Drive, before the Colab runtime is recycled.
from google.colab import files
files.download(DEEP_FILE)